In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "qs-cell-1",
   "metadata": {},
   "outputs": [],
   "source": [
    "#@title 1. Clone repository and verify dependencies\n",
    "import os\n",
    "import subprocess\n",
    "import sys\n",
    "from pathlib import Path\n",
    "\n",
    "# Anonymized companion repository URL for peer review\n",
    "REPO_URL = \"https://github.com/edu-sandbox/cer-genai-multioutcome-power.git\"  #@param {type:\"string\"}\n",
    "REPO_DIR = \"cer_genai_power_sim\"\n",
    "\n",
    "def _run(cmd):\n",
    "    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n",
    "    assert proc.stdout is not None\n",
    "    for line in proc.stdout:\n",
    "        print(line, end=\"\", flush=True)\n",
    "    rc = proc.wait()\n",
    "    if rc != 0:\n",
    "        raise subprocess.CalledProcessError(rc, cmd)\n",
    "\n",
    "if not REPO_URL.startswith(\"https://\") or \"<\" in REPO_URL:\n",
    "    raise ValueError(\n",
    "        \"Please provide a valid public GitHub repository URL \"\n",
    "        \"(or upload the project .py files into the working directory).\"\n",
    "    )\n",
    "\n",
    "# Idempotent directory handling so re-running Cell 1 in the same session never fails\n",
    "if not Path(\"variant_runner_safe.py\").exists():\n",
    "    if not Path(REPO_DIR).exists():\n",
    "        _run([\"git\", \"clone\", \"--depth\", \"1\", REPO_URL, REPO_DIR])\n",
    "    os.chdir(REPO_DIR)\n",
    "\n",
    "# Ensure required scientific Python packages are installed\n",
    "_run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"numpy\", \"pandas\", \"scipy\", \"statsmodels\", \"matplotlib\"])\n",
    "\n",
    "# Run built-in self-tests (Frechet-Boole bounds, Weibull/Student-t transforms, bisection guard)\n",
    "_run([sys.executable, \"variant_runner_safe.py\", \"--selftest\"])"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "qs-cell-2",
   "metadata": {},
   "outputs": [],
   "source": [
    "#@title 2. Configure pilot effect sizes and run --quickstart (synthetic data only)\n",
    "import subprocess\n",
    "import sys\n",
    "\n",
    "# PRIVACY NOTE: --quickstart takes only four aggregate effect-size numbers and\n",
    "# simulates strictly synthetic cohorts. Never upload real student data to Colab.\n",
    "PILOT_HR = 1.80       #@param {type:\"number\"}\n",
    "PILOT_D_EL = -0.40    #@param {type:\"number\"}\n",
    "PILOT_D_GL = -0.30    #@param {type:\"number\"}\n",
    "PILOT_D_TRANS = -0.50 #@param {type:\"number\"}\n",
    "TARGET_POWER = 0.80   #@param {type:\"number\"}\n",
    "VARIANT = \"base\"      #@param [\"base\", \"W\", \"A\", \"F\", \"C\", \"T\"]\n",
    "REPS = 500            #@param {type:\"integer\"}\n",
    "WORKERS = 2           #@param {type:\"integer\"}\n",
    "\n",
    "cmd = [\n",
    "    sys.executable, \"-u\", \"variant_runner_safe.py\", \"--quickstart\",\n",
    "    \"--pilot-hr\", str(PILOT_HR),\n",
    "    \"--pilot-d-el\", str(PILOT_D_EL),\n",
    "    \"--pilot-d-gl\", str(PILOT_D_GL),\n",
    "    \"--pilot-d-trans\", str(PILOT_D_TRANS),\n",
    "    \"--power\", str(TARGET_POWER),\n",
    "    \"--variant\", VARIANT,\n",
    "    \"--reps\", str(REPS),\n",
    "    \"--workers\", str(WORKERS),\n",
    "]\n",
    "print(\"Running:\", \" \".join(cmd), flush=True)\n",
    "proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n",
    "assert proc.stdout is not None\n",
    "for line in proc.stdout:\n",
    "    print(line, end=\"\", flush=True)\n",
    "rc = proc.wait()\n",
    "if rc != 0:\n",
    "    raise subprocess.CalledProcessError(rc, cmd)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "qs-cell-3",
   "metadata": {},
   "outputs": [],
   "source": [
    "#@title 3. Inspect required sample sizes (N_req) and audit Frechet-Boole bounds\n",
    "import subprocess\n",
    "import sys\n",
    "import pandas as pd\n",
    "from pathlib import Path\n",
    "\n",
    "out_dir = Path(\"variant_output_safe\")\n",
    "grid_path = out_dir / \"quickstart_grid.csv\"\n",
    "res_path = out_dir / \"quickstart_result.csv\"\n",
    "\n",
    "# Verify Frechet-Boole bounds on the generated grid\n",
    "audit = subprocess.run(\n",
    "    [sys.executable, \"variant_runner_safe.py\", \"--check-csv\", str(grid_path)],\n",
    "    capture_output=True, text=True, check=True,\n",
    ")\n",
    "print(audit.stdout.strip())\n",
    "\n",
    "df_req = pd.read_csv(res_path, encoding=\"utf-8-sig\")\n",
    "df_grid = pd.read_csv(grid_path, encoding=\"utf-8-sig\")\n",
    "\n",
    "print(\"\\n=== Interpolated Required Sample Sizes (N_req) ===\")\n",
    "print(df_req.to_string(index=False))\n",
    "\n",
    "cols = [\n",
    "    \"N\", \"v1_pow_H1a\", \"v2_pow_H1a\", \"v3_pow_H1a\",\n",
    "    \"v3_pow_H2a_EL\", \"v3_pow_H2b_GL\", \"v3_pow_H3\", \"v3_joint\",\n",
    "    \"tost_unpaired\", \"tost_paired\", \"early_cens_%\",\n",
    "]\n",
    "present = [c for c in cols if c in df_grid.columns]\n",
    "print(\"\\n=== Empirical Power Grid ===\")\n",
    "print(df_grid[present].round(3).to_string(index=False))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "qs-cell-4",
   "metadata": {},
   "outputs": [],
   "source": [
    "#@title 4. Plot multi-outcome power curves and Cox estimator comparison\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "from pathlib import Path\n",
    "\n",
    "out_dir = Path(\"variant_output_safe\")\n",
    "df_grid = pd.read_csv(out_dir / \"quickstart_grid.csv\", encoding=\"utf-8-sig\").sort_values(\"N\")\n",
    "df_req = pd.read_csv(out_dir / \"quickstart_result.csv\", encoding=\"utf-8-sig\").iloc[0]\n",
    "target_p = float(df_req.get(\"target_power\", 0.80))\n",
    "\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=130)\n",
    "\n",
    "# Panel (a): Endpoint-specific and joint power curves (v3) with +/-1.96 MCSE ribbons\n",
    "curves = [\n",
    "    (\"v3_pow_H1a\", \"v3_mcse_H1a\", \"H1a: Speed (Cox v3)\", \"o-\"),\n",
    "    (\"v3_pow_H3\", \"v3_mcse_H3\", \"H3: Transfer (ANCOVA)\", \"s-\"),\n",
    "    (\"v3_pow_H2a_EL\", \"v3_mcse_H2a_EL\", \"H2a: Extraneous load\", \"^-\"),\n",
    "    (\"v3_pow_H2b_GL\", \"v3_mcse_H2b_GL\", \"H2b: Germane load\", \"v-\"),\n",
    "    (\"v3_joint\", \"v3_joint_mcse\", \"Joint (all 4 confirmed)\", \"d--\"),\n",
    "]\n",
    "for col, mcol, label, fmt in curves:\n",
    "    y = df_grid[col].values\n",
    "    se = df_grid[mcol].values if mcol in df_grid.columns else np.zeros_like(y)\n",
    "    line, = ax1.plot(df_grid[\"N\"], y, fmt, label=label, linewidth=2)\n",
    "    ax1.fill_between(\n",
    "        df_grid[\"N\"],\n",
    "        np.clip(y - 1.96 * se, 0.0, 1.0),\n",
    "        np.clip(y + 1.96 * se, 0.0, 1.0),\n",
    "        color=line.get_color(),\n",
    "        alpha=0.14,\n",
    "    )\n",
    "\n",
    "ax1.plot(df_grid[\"N\"], df_grid[\"tost_unpaired\"], \"D--\", alpha=0.8, label=\"H1b: Equivalence (TOST)\")\n",
    "ax1.axhline(target_p, color=\"gray\", linestyle=\":\", linewidth=1.2)\n",
    "ax1.set_title(\"(a) Multi-Outcome Power Curves (v3)\")\n",
    "ax1.set_xlabel(\"Total sample size (N)\")\n",
    "ax1.set_ylabel(\"Empirical power (+/- 1.96 MCSE)\")\n",
    "ax1.set_ylim(-0.02, 1.04)\n",
    "ax1.grid(alpha=0.25)\n",
    "ax1.legend(fontsize=8, loc=\"best\")\n",
    "\n",
    "# Panel (b): Naive v1 vs. Hybrid Jackknife v2 vs. Drop-Early v3 + Early Censoring %\n",
    "ax2.plot(df_grid[\"N\"], df_grid[\"v3_pow_H1a\"], \"o-\", linewidth=2, label=\"v3: Drop early cens. + sandwich\")\n",
    "if \"v2_pow_H1a\" in df_grid.columns:\n",
    "    ax2.plot(df_grid[\"N\"], df_grid[\"v2_pow_H1a\"], \"s-.\", linewidth=1.8, label=\"v2: Hybrid + cluster jackknife\")\n",
    "if \"v1_pow_H1a\" in df_grid.columns:\n",
    "    ax2.plot(df_grid[\"N\"], df_grid[\"v1_pow_H1a\"], \"^--\", linewidth=1.8, label=\"v1: Naive sandwich (NaN -> p=1)\")\n",
    "ax2.axhline(target_p, color=\"gray\", linestyle=\":\", linewidth=1.2)\n",
    "ax2.set_title(\"(b) Cox Estimator Comparison & Early Censoring\")\n",
    "ax2.set_xlabel(\"Total sample size (N)\")\n",
    "ax2.set_ylabel(\"H1a Empirical Power (Holm-adjusted)\")\n",
    "ax2.set_ylim(0.0, 1.04)\n",
    "ax2.grid(alpha=0.25)\n",
    "\n",
    "h1, l1 = ax2.get_legend_handles_labels()\n",
    "if \"early_cens_%\" in df_grid.columns:\n",
    "    ax2_r = ax2.twinx()\n",
    "    ax2_r.bar(df_grid[\"N\"], df_grid[\"early_cens_%\"], width=8, alpha=0.18, color=\"gray\", label=\"Early censoring (%)\")\n",
    "    ax2_r.set_ylabel(\"Early censoring / v1 NaN SE rate (%)\")\n",
    "    ax2_r.set_ylim(0, max(25.0, float(df_grid[\"early_cens_%\"].max()) * 1.4))\n",
    "    h2, l2 = ax2_r.get_legend_handles_labels()\n",
    "    ax2.legend(h1 + h2, l1 + l2, fontsize=8, loc=\"lower right\")\n",
    "else:\n",
    "    ax2.legend(h1, l1, fontsize=8, loc=\"lower right\")\n",
    "\n",
    "plt.tight_layout()\n",
    "out_fig = out_dir / \"quickstart_power_plot.png\"\n",
    "plt.savefig(out_fig, dpi=150)\n",
    "print(f\"Saved figure: {out_fig}\")\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "colab": {
   "name": "quickstart_colab.ipynb",
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}